# Module 8 · Solutions

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
BASE = "data/"
fin = pd.read_csv(BASE + "company_financials.csv"); latest = fin.iloc[-1]
TAX, REINVEST = 0.252, 0.045
fcf0 = latest["ebit_cr"]*(1-TAX) + latest["depreciation_cr"] - REINVEST*latest["revenue_cr"]
years = np.arange(1,6)

In [ ]:
# Ex1 - the Gordon cliff
def dcf_ev(wacc, g, growth=0.12):
    fcf = fcf0*(1+growth)**years
    return (fcf/(1+wacc)**years).sum() + fcf[-1]*(1+g)/(wacc-g)/(1+wacc)**5
print(f"WACC 11.5%, g 5.0%: Rs {dcf_ev(0.115, 0.05):,.0f} cr")
print(f"WACC  6.5%, g 6.0%: Rs {dcf_ev(0.065, 0.06):,.0f} cr   <- explodes")
print("\nGordon TV = FCF x (1+g) / (WACC - g). As g -> WACC the denominator -> 0: value -> infinity.")
print("The formula assumes growth FOREVER below the discount rate; violate that and the arithmetic")
print("dutifully prices an impossible company. Any DCF where (WACC - g) < ~3-4pp is standing on this cliff -")
print("its terminal value is hypersensitive and the range, not the point, is the only honest output.")

In [ ]:
# Ex2 - regression comps
peers = pd.DataFrame({"ev_ebitda":[14.5,11.8,16.2,12.9,13.6], "g":[0.14,0.09,0.18,0.11,0.12]})
r = stats.linregress(peers["g"], peers["ev_ebitda"])
implied = r.intercept + r.slope*0.12
print(f"Fit: EV/EBITDA = {r.intercept:.1f} + {r.slope:.0f} x growth   (R2 {r.rvalue**2:.2f}, n=5!)")
print(f"At MoneyMart's 12% growth: implied multiple {implied:.1f}x -> EV Rs {implied*latest['ebitda_cr']:,.0f} cr")
print("\nJustifies placing MoneyMart mid-range rather than at the median blindly. With 5 points this is")
print("a sketch - banks run it with 15-30 peers - but the LOGIC (multiples are priced growth) is exact.")

In [ ]:
# Ex3 - recession football field
peers_full = pd.DataFrame({"ev_ebitda":[14.5,11.8,16.2,12.9,13.6], "pe":[28.0,22.5,34.0,25.5,26.8]}) * 0.8
ebitda, pat, NET_DEBT = latest["ebitda_cr"], latest["pat_cr"], 600
ee_lo, ee_hi = peers_full.ev_ebitda.quantile(.25)*ebitda, peers_full.ev_ebitda.quantile(.75)*ebitda
pe_lo, pe_hi = peers_full.pe.quantile(.25)*pat + NET_DEBT, peers_full.pe.quantile(.75)*pat + NET_DEBT
waccs, gs = [0.105,0.11,0.115,0.12,0.125], [0.04,0.045,0.05,0.055,0.06]
grid = np.array([[dcf_ev(w,g,growth=0.08) for g in gs] for w in waccs])
methods = [("DCF", grid.min(), grid.max()), ("EV/EBITDA", ee_lo, ee_hi), ("P/E", pe_lo, pe_hi)]
for n, lo, hi in methods: print(f"{n:<10} Rs {lo:8,.0f} - {hi:8,.0f} cr")
ov_lo, ov_hi = max(m[1] for m in methods), min(m[2] for m in methods)
print(f"\nConvergence zone: {'Rs %s-%s cr' % (f'{ov_lo:,.0f}', f'{ov_hi:,.0f}') if ov_lo < ov_hi else 'GONE - methods no longer agree'}")
print("In a de-rating, comps fall with the market's mood while the DCF falls with YOUR growth cut -")
print("they can decouple. A vanishing convergence zone is the chart telling the deal team: wide bid-ask,")
print("deals stall, sellers anchor to yesterday's multiples. Timing IS a valuation output.")